## Test automation

- Use local tracing to help your debug and test your application
- Separate prompts, sample data for testing from the code

#### Prompty
Prompty is a file with .prompty extension for developing prompt template. 
The prompty asset is a markdown file with a modified front matter. 
The front matter is in yaml format that contains a number of metadata fields which defines model configuration and expected inputs of the prompty.

#### Create necessary connections
Connection helps securely store and manage secret keys or other sensitive credentials required for interacting with LLM and other external tools for example Azure Content Safety.

Above prompty uses connection `open_ai_connection` inside, we need to set up the connection if we haven't added it before. After created, it's stored in local db and can be used in any flow.

Prepare your Azure OpenAI resource follow this [instruction](https://learn.microsoft.com/en-us/azure/cognitive-services/openai/how-to/create-resource?pivots=web-portal) and get your `api_key` if you don't have one.

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_KEY = os.getenv("AZURE_OPENAI_KEY")
AZURE_OPENAI_GPT4_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_GPT4_DEPLOYMENT_NAME")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")

In [2]:
from promptflow.client import PFClient
from promptflow.connections import AzureOpenAIConnection, OpenAIConnection

# client can help manage your runs and connections.
pf = PFClient()
try:
    conn_name = "open_ai_connection"
    conn = pf.connections.get(name=conn_name)
    print("using existing connection")
except:
    # Follow https://learn.microsoft.com/en-us/azure/ai-services/openai/how-to/create-resource?pivots=web-portal to create an Azure OpenAI resource.
    connection = AzureOpenAIConnection(
        name=conn_name,
        api_key=AZURE_OPENAI_KEY,
        api_base=AZURE_OPENAI_ENDPOINT,
        api_type="azure",
    )


    conn = pf.connections.create_or_update(connection)
    print("successfully created connection")

print(conn)

using existing connection
auth_mode: key
name: open_ai_connection
module: promptflow.connections
created_date: '2025-01-02T13:57:44.224476'
last_modified_date: '2025-01-05T15:51:28.594240'
type: azure_open_ai
api_key: '******'
api_base: https://ai-hubpfrag458423774142.openai.azure.com/
api_type: azure
api_version: '2024-02-01'



In [3]:
from promptflow.core import Prompty

# load prompty as a flow
f = Prompty.load("./prompts/chat.prompty")
# execute the flow as function
question = "What is the capital of France?"
result = f(question=question)
result

'The capital of France is Paris! 🗼✨'

## 2. Batch run with multi-line data  

Use the data in data.jsonl file for batch testing our prompt in the chat.prompty file


In [4]:
from promptflow.client import PFClient

flow = "./prompts/chat.prompty"  # path to the prompty file
data = "./data/data.jsonl"  # path to the data file

# create run with the flow and data
pf = PFClient()
base_run = pf.run(
    flow=flow,
    data=data,
    column_mapping={
        "question": "${data.question}",
        "chat_history": "${data.chat_history}",
    },
    stream=True,
)

Starting prompt flow service...


ERROR:azure.monitor.opentelemetry.exporter.export._base:Non-retryable server side error: Operation returned an invalid status 'Bad Request'.
ERROR:azure.monitor.opentelemetry.exporter.export._base:Non-retryable server side error: Operation returned an invalid status 'Bad Request'.


[2025-01-16 09:57:18 +0200][promptflow._sdk._orchestrator.run_submitter][INFO] - Submitting run prompts_20250116_095704_952890, log path: C:\Users\dschlesinger\.promptflow\.runs\prompts_20250116_095704_952890\logs.txt


You can stop the prompt flow service with the following command:'pf service stop'.

You can view the traces in local from http://127.0.0.1:23333/v1.0/ui/traces/?#run=prompts_20250116_095704_952890
2025-01-16 09:57:18 +0200   16876 execution.bulk     INFO     Current thread is not main thread, skip signal handler registration in BatchEngine.
2025-01-16 09:57:19 +0200   16876 execution.bulk     INFO     Current system's available memory is 30261.796875MB, memory consumption of current process is 204.4140625MB, estimated available worker count is 30261.796875/204.4140625 = 148
2025-01-16 09:57:19 +0200   16876 execution.bulk     INFO     Set process count to 3 by taking the minimum value among the factors of {'default_worker_count': 4, 'row_count': 3, 'estimated_worker_count_based_on_memory_usage': 148}.
2025-01-16 09:57:23 +0200   16876 execution.bulk     INFO     Process name(SpawnProcess-2)-Process id(28020)-Line number(0) start execution.
2025-01-16 09:57:23 +0200   16876 execution.bu

In [5]:
details = pf.get_details(base_run)
details.head(10)

,inputs.question,inputs.chat_history,inputs.line_number,outputs.output
0,What's chat-GPT?,[],0,Chat-GPT is an advanced language model develop...
1,How many questions did John Doe ask?,[],1,"I'm sorry, but I don't have access to specific..."
2,How many questions did John Doe ask?,"[{'role': 'user', 'content': 'where is the nea...",2,I'm not sure which John Doe you're referring t...
